In [2]:
"""
collect_trajectories.py
=======================
Generates a static offline RL dataset by running TokaMaker_TORAX simulations
with varied ECRH and NBI heating schedules sampled via Latin Hypercube Sampling.

Dataset structure (per trajectory, 12 transitions):
    s       : state vector at rl_time[i]
    a       : [ecrh_MW, nbi_MW] applied from rl_time[i] to rl_time[i+1]
    r       : mean Q_fusion from rl_time[i] to rl_time[i+1]
              (plus terminal reward at last step)
    s_next  : state vector at rl_time[i+1]
    done    : True at last transition

Usage:
    python collect_trajectories.py --n_trajectories 500 --output_dir ./rl_dataset
"""

import os
import json
import time
import argparse
import numpy as np
from scipy.stats import qmc
from datetime import datetime
import io
from contextlib import redirect_stdout

# ── RL / simulation config ────────────────────────────────────────────────────

# Times at which RL states, actions, and rewards are collected
ALL_TIMES = [0, 50, 70, 80, 130, 180, 230, 280, 330, 380, 450, 520, 580, 600]
RL_TIMES = [50, 70, 80, 130, 180, 230, 280, 330, 380, 450, 520, 580, 600]

# Decision times = all RL times except the final state-only point
DECISION_TIMES = RL_TIMES[:-1]  # [50, 70, ..., 580]

# TokaMaker solve times (fine resolution for physics accuracy)
N_TM_POINTS = 10
TM_TIMES = np.linspace(0, 600, N_TM_POINTS)

# Action bounds
ECRH_MIN, ECRH_MAX = 0.0, 40.0   # MW
NBI_MIN,  NBI_MAX  = 0.0, 33.0   # MW

# NBI is physically off before L-H transition
NBI_ZERO_BEFORE = 80  # seconds

# Safety thresholds for penalty
# source for safety thresholds: https://www.iter.org/sites/default/files/education/L02_Wagner.pdf
# another source for safety thresholds: https://www.osti.gov/servlets/purl/6227385
Q95_MIN     = 3.0
BETA_N_MAX  = 2.8
FGW_MAX     = 0.85
SAFETY_PENALTY = 50.0  # subtracted from reward at any violated timestep

# Terminal reward weight on flux consumption
FLUX_WEIGHT = 0.1  # R_terminal = Q_flattop_avg - FLUX_WEIGHT * flux_consumed_Wb

# Pellet schedule (fixed to baseline)
PELLET_S_TOTAL = {0: 0, 90: 5e21, 450: 5e21, 451: 0}

# ── Latin Hypercube Sampling ──────────────────────────────────────────────────

def sample_actions_lhs(n_trajectories, seed=42):
    """
    Sample heating schedules using Latin Hypercube Sampling.
    
    Returns array of shape (n_trajectories, n_decision_times, 2)
    where axis -1 is [ecrh_MW, nbi_MW].
    
    NBI is forced to 0 at decision times before NBI_ZERO_BEFORE.
    """
    n_decision = len(DECISION_TIMES)
    n_params = n_decision * 2  # ecrh + nbi at each decision time

    sampler = qmc.LatinHypercube(d=n_params, seed=seed)
    samples = sampler.random(n=n_trajectories)  # shape (N, n_params)

    # Scale to physical bounds
    lower = np.array([ECRH_MIN, NBI_MIN] * n_decision)
    upper = np.array([ECRH_MAX, NBI_MAX] * n_decision)
    samples_scaled = qmc.scale(samples, lower, upper)  # shape (N, n_params)

    # Reshape to (N, n_decision_times, 2)
    actions = samples_scaled.reshape(n_trajectories, n_decision, 2)

    # Enforce NBI=0 before L-H transition
    for i, t in enumerate(DECISION_TIMES):
        if t < NBI_ZERO_BEFORE:
            actions[:, i, 1] = 0.0

    return actions  # MW units


# ── Schedule builders ─────────────────────────────────────────────────────────

def build_ecrh_schedule(action_row):
    """
    Build ecrh_powers dict from action_row (n_decision_times, 2).
    Appends t=600: 0 for safe shutdown.
    action_row[:, 0] = ecrh_MW at each decision time.
    """
    schedule = {}
    for i, t in enumerate(DECISION_TIMES):
        schedule[t] = float(action_row[i, 0]) * 1e6  # convert MW -> W
    schedule[600] = 0.0
    return schedule


def build_nbi_schedule(action_row):
    """
    Build generic_powers dict from action_row.
    NBI is 0 before t=80 by construction from sampling.
    action_row[:, 1] = nbi_MW at each decision time.
    """
    schedule = {0: 0.0, 79: 0.0}  # ensure 0 before L-H
    for i, t in enumerate(DECISION_TIMES):
        if t >= NBI_ZERO_BEFORE:
            schedule[t] = float(action_row[i, 1]) * 1e6  # MW -> W
    schedule[600] = 0.0
    return schedule


# ── State extraction ──────────────────────────────────────────────────────────

def get_nearest_tm_index(rl_time, tm_times):
    """Return index of closest TM timepoint to rl_time."""
    return int(np.argmin(np.abs(np.array(tm_times) - rl_time)))


def interpolate_torax_scalar(tmtx, var_name, rl_time):
    """Interpolate a TORAX DataTree scalar to a specific rl_time."""
    try:
        torax_times = tmtx._data_tree['scalars'].coords['time'].values
        values = tmtx._data_tree['scalars'][var_name].values.astype(float)
        return float(np.interp(rl_time, torax_times, values))
    except Exception:
        return float('nan')


def get_torax_profile_at_rho(tmtx, var_name, rho_values, rl_time):
    """
    Interpolate a TORAX profile variable to specific rho points at rl_time.
    Returns list of floats, one per rho value.
    """
    ds = tmtx._data_tree['profiles']
    torax_times = ds.coords['time'].values

    # Find nearest time index in TORAX
    t_idx = int(np.argmin(np.abs(torax_times - rl_time)))
    profile_data = ds[var_name].values  # shape (time, rho)

    # Check which rho coordinate THIS SPECIFIC VARIABLE uses
    var_dims = ds[var_name].dims
    print(f"DEBUG: var={var_name}, dims={var_dims}")  # DIAGNOSTIC
    
    if 'rho_face_norm' in var_dims:
        rho_coord = ds.coords['rho_face_norm'].values
        print(f"  -> using rho_face_norm, shape={rho_coord.shape}")
    elif 'rho_norm' in var_dims:
        rho_coord = ds.coords['rho_norm'].values
        print(f"  -> using rho_norm, shape={rho_coord.shape}")
    elif 'rho_cell_norm' in var_dims:
        rho_coord = ds.coords['rho_cell_norm'].values
        print(f"  -> using rho_cell_norm, shape={rho_coord.shape}")
    else:
        print(f"  -> ERROR: no rho coordinate found!")
        return [float('nan')] * len(rho_values)

    profile_at_t = profile_data[t_idx, :]
    print(f"  profile_at_t shape={profile_at_t.shape}, first 5 vals={profile_at_t[:5]}")
    
    result = [float(np.interp(rho, rho_coord, profile_at_t)) for rho in rho_values]
    print(f"  result for rho={rho_values}: {result}")
    return result


def extract_state(tmtx, t_start, t_end):
    """
    Extract state vector for interval [t_start, t_end].
    
    Returns dict with all state quantities. Scalars are taken at t_start.
    Q_fusion is the mean over the interval [t_start, t_end].
    """
    tm_idx = get_nearest_tm_index(t_start, tmtx._tm_times)
    s = tmtx._state
    rho_points = [0.2, 0.5, 0.8]
    state = {}

    # ── Scalars from TORAX DataTree at t_start ────────────────────────────────
    torax_scalars = [
        'H98', 
        'tau_E', 
        'W_thermal_total',
        'P_SOL_total', 
        'P_radiation_e', 
        'P_aux_total',
        'f_non_inductive', 
        'n_e_line_avg', 
        'fgw_n_e_line_avg',
        'T_e_volume_avg', 
        'T_i_volume_avg', 
        'beta_N', 
        'li3',
        'dW_thermal_dt_smoothed', 
        'P_ohmic_e', 
        'q_min',
        'f_bootstrap', 
        'P_alpha_total',
        'q95',
        'v_loop_lcfs',
        'Ip'
    ]
    for var in torax_scalars:
        state[f'tx_{var}'] = interpolate_torax_scalar(tmtx, var, t_start)

    # ── Q_fusion: mean over interval ──────────────────────────────────────────
    torax_times = tmtx._data_tree['scalars'].coords['time'].values
    mask = (torax_times >= t_start) & (torax_times <= t_end)
    if np.any(mask):
        Q_vals = tmtx._data_tree['scalars']['Q_fusion'].values[mask]
        state['tx_Q_fusion'] = float(np.nanmean(Q_vals))
    else:
        state['tx_Q_fusion'] = 0.0

    # ── Profiles at rho = 0.2, 0.5, 0.8 at t_start ────────────────────────────
    profile_vars = ['T_e', 'T_i', 'n_e', 'q']
    for var in profile_vars:
        vals = get_torax_profile_at_rho(tmtx, var, rho_points, t_start)
        for rho, val in zip(rho_points, vals):
            state[f'{var}_rho{int(rho*10)}'] = val

    return state


# ── Reward computation ────────────────────────────────────────────────────────

def compute_reward(tmtx, t_start, t_end, is_terminal=False):
    """
    Compute reward for the interval [t_start, t_end].
    
    Step reward: mean Q_fusion over the interval.
    Safety penalty: applied if any safety threshold violated in interval.
    Terminal bonus (last step only): Q_flattop_avg - FLUX_WEIGHT * flux_consumed.
    """
    torax_times = tmtx._data_tree['scalars'].coords['time'].values

    # Mask for this interval
    mask = (torax_times >= t_start) & (torax_times <= t_end)
    if not np.any(mask):
        step_reward = 0.0
    else:
        Q_vals = tmtx._data_tree['scalars']['Q_fusion'].values[mask]
        step_reward = float(np.nanmean(Q_vals))

    # Safety penalties over this interval
    penalty = 0.0

    if t_start >= 80:
        # apply penalties
        try:
            q95_vals   = tmtx._data_tree['scalars']['q95'].values[mask]
            betaN_vals = tmtx._data_tree['scalars']['beta_N'].values[mask]
            fgw_vals   = tmtx._data_tree['scalars']['fgw_n_e_line_avg'].values[mask]

            if np.any(q95_vals < Q95_MIN):
                penalty += SAFETY_PENALTY
            if np.any(betaN_vals > BETA_N_MAX):
                penalty += SAFETY_PENALTY
            if np.any(fgw_vals > FGW_MAX):
                penalty += SAFETY_PENALTY
        except Exception:
            pass

    reward = step_reward - penalty

    # Terminal bonus
    if is_terminal:

        with redirect_stdout(io.StringIO()):
            summary = tmtx.summary()

        Q_avg   = summary.get('Q_flattop_avg', 0.0)
        flux_wb = summary.get('flux_consumed_Wb', 0.0)
        reward += Q_avg - FLUX_WEIGHT * flux_wb

    return reward


# ── Trajectory builder ────────────────────────────────────────────────────────

def build_trajectory(tmtx, action_row):
    """
    After fly() has completed, extract the full list of transitions.
    
    Returns list of dicts, each with keys: s, a, r, s_next, done.
    Length = len(DECISION_TIMES) = 12.
    """
    transitions = []

    for i, t in enumerate(DECISION_TIMES):
        t_next = RL_TIMES[i + 1]
        is_terminal = (i == len(DECISION_TIMES) - 1)

        s      = extract_state(tmtx, t, t_next)
        a      = action_row[i].tolist()  # [ecrh_MW, nbi_MW]
        r      = compute_reward(tmtx, t, t_next, is_terminal=is_terminal)

        transitions.append({
            's':      s,
            'a':      a,          # [ecrh_MW, nbi_MW] in MW
            'r':      r,
            'done':   is_terminal,
            't':      t,
            't_next': t_next,
        })

    return transitions


# ── Main simulation setup ─────────────────────────────────────────────────────

def setup_tokamaker(cwd):
    """Initialize OFT and TokaMaker, produce seed eqdsks. Run once."""

    import sys
    sys.path.append('/Users/deniz/Desktop/Spring2026/CS224R/project/OpenFUSIONToolkit/install_release/python')  
    from OpenFUSIONToolkit import OFT_env
    from OpenFUSIONToolkit.TokaMaker import TokaMaker
    from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
    from OpenFUSIONToolkit.TokaMaker.util import create_power_flux_fun, create_isoflux
    import numpy as np

    R0, B0, Z0 = 6.3, 5.2, 0.5

    myOFT = OFT_env(nthreads=2)
    mygs  = TokaMaker(myOFT)

    mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh('ITER_mesh.h5')
    mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
    mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)
    mygs.settings.maxits = 100
    mygs.setup(order=2, F0=R0 * B0)
    mygs.set_coil_vsc({'VS': 1.0})

    return mygs, R0, B0, Z0


def run_single_trajectory(mygs, action_row, run_id, cwd, eqdsk_list, eqtimes,
                           coil_bounds, x_points, diverted_isoflux_pts,
                           Ip_targets, ne_init, Te_init, psi_sample):
    """
    Configure and run one TokaMaker_TORAX simulation with the given action_row.
    Returns the transitions list and summary dict, or None if simulation failed.
    """
    from OpenFUSIONToolkit.TokaMaker.pulse_design import TokaMaker_TORAX
    import numpy as np

    ecrh_schedule = build_ecrh_schedule(action_row)
    nbi_schedule  = build_nbi_schedule(action_row)

    try:
        tmtx = TokaMaker_TORAX(
            t_init=0,
            t_final=600,
            tx_dt=5,
            eqtimes=eqtimes,
            g_eqdsk_arr=eqdsk_list,
            last_surface_factor=0.99,
            tm_times=TM_TIMES,
            tokamaker_obj=mygs,
        )

        tmtx.set_TORAX_grid(grid_type='n_rho', grid=10)

        tmtx.set_heating(
            generic_heat=nbi_schedule,
            generic_heat_loc=0.25,
            nbi_current=True,
            ecrh=ecrh_schedule,
            ecrh_loc=0.35,
        )

        tmtx.set_fueling(
            gas_puff_S_total=1e22,
            gas_puff_decay_length=0.05,
            pellet_deposition_location=0.8,
            pellet_width=0.1,
            pellet_S_total=PELLET_S_TOTAL,
        )

        def array_to_profile_dict(arr, grid):
            return {float(p): float(v) for p, v in zip(grid, arr)}

        ne = {0.0: array_to_profile_dict(ne_init, psi_sample)}
        Te = {0.0: array_to_profile_dict(Te_init, psi_sample)}

        tmtx.set_ne(ne, right_bc={0: ne_init[-1], 80: 2e19, 500: 2e19, 600: 0.5e19})
        tmtx.set_Te(Te, right_bc=0.1)
        tmtx.set_Ti(Te, right_bc=0.1)

        ne_ped_val, Te_ped_val = 0.9e20, 3.0
        ped_toggle = {0: False, 79: False, 80: True, 500: True, 501: False, 600: False}
        T_ped  = {80: 1.0, 82: 2.0, 90: Te_ped_val, 500: Te_ped_val, 540: 1.0, 580: 1.0, 600: 1.0}
        n_e_ped = {80: 3e19, 82: ne_ped_val / 2, 90: ne_ped_val, 500: ne_ped_val}
        tmtx.set_pedestal(set_pedestal=ped_toggle, T_i_ped=T_ped, T_e_ped=T_ped,
                          n_e_ped=n_e_ped, ped_top=0.9)

        tmtx.set_Ip({0: Ip_targets[0], 100: Ip_targets[2], 500: Ip_targets[2], 600: Ip_targets[0]})
        tmtx.set_plasma_composition(main_ion={'D': 0.5, 'T': 0.5}, impurity='Ne', Zeff=1.6)
        tmtx.set_evolve(density=True, Ti=True, Te=True, current=True)
        tmtx.set_x_points(diverted_times=(80, 500), x_point_targets=x_points, x_point_weight=100)
        tmtx.set_TokaMaker_coil_reg(coil_bounds=coil_bounds, updownsym=False)

        tmtx.fly(
            output_mode=False,
            max_loop=2,
            run_name='tmp',
            t_ave_toggle='flattop',
            t_ave_window=25,
            relax=True,
            relax_duration=5,
        )

        transitions = build_trajectory(tmtx, action_row)
        with redirect_stdout(io.StringIO()):
            summary = tmtx.summary()

        return transitions, summary

    except Exception as e:
        print(f'  [run {run_id}] FAILED: {e}')
        return None, None


# ── Dataset saving ────────────────────────────────────────────────────────────

def save_trajectory(transitions, summary, action_row, run_id, output_dir):
    """Save one trajectory as a JSON file."""
    payload = {
        'run_id':      run_id,
        'timestamp':   datetime.now().isoformat(),
        'actions_raw': action_row.tolist(),   # (12, 2) array in MW
        'transitions': transitions,            # list of 12 dicts
        'summary':     summary,
    }
    path = os.path.join(output_dir, f'trajectory_{run_id:04d}.json')
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2, default=str)
    return path



In [ ]:
n_trajectories = 600
output_dir     = './rl_dataset_test'
seed           = 42
start_idx      = 0

os.makedirs(output_dir, exist_ok=True)
cwd = os.getcwd()

print(f'Sampling {n_trajectories} trajectories with LHS (seed={seed})')
all_actions = sample_actions_lhs(n_trajectories, seed=seed)
np.save(os.path.join(output_dir, 'all_actions.npy'), all_actions)
print(f'Action matrix saved: shape {all_actions.shape}')

# ── One-time setup ────────────────────────────────────────────────────────
print('Setting up TokaMaker...')
mygs, R0, B0, Z0 = setup_tokamaker(cwd)

# These are fixed across all runs (same as baseline notebook)
Ip_targets = [1.5e6, 5e6, 15e6, 15e6, 1.5e6]
eqdsk_list = [os.path.join(cwd, f'i={i}.eqdsk') for i in range(5)]
eqtimes    = [0, 30, 80, 500, 600]
x_points   = np.array([[5.125, -3.4]])
diverted_isoflux_pts = np.array([
    [8.20, 0.41], [8.06, 1.46], [7.51, 2.62], [6.14, 3.78],
    [5.10, 3.72], [4.51, 3.02], [4.26, 1.33], [4.28, 0.08],
    [4.49, -1.34], [7.28, -1.89], [8.00, -0.68]
])
coil_bounds = {key: [-50.e6, 50.e6] for key in mygs.coil_sets}

psi_sample = np.linspace(0.0, 1.0, 25)
ne_init = np.array([3.00e+19, 2.73e+19, 2.49e+19, 2.28e+19, 2.09e+19,
                    1.92e+19, 1.78e+19, 1.65e+19, 1.54e+19, 1.44e+19,
                    1.35e+19, 1.27e+19, 1.20e+19, 1.14e+19, 1.09e+19,
                    1.04e+19, 9.98e+18, 9.61e+18, 9.29e+18, 9.00e+18,
                    8.75e+18, 8.52e+18, 8.33e+18, 8.15e+18, 8.00e+18])
Te_init = np.array([1.50, 1.33, 1.17, 1.04, 0.92, 0.82, 0.72, 0.64,
                    0.57, 0.50, 0.45, 0.40, 0.36, 0.32, 0.28, 0.25,
                    0.23, 0.20, 0.18, 0.16, 0.15, 0.13, 0.12, 0.11, 0.10])

# ── Run loop ──────────────────────────────────────────────────────────────
success_count = 0
fail_count    = 0
t_start_total = time.time()

for run_id in range(start_idx, n_trajectories):
    action_row = all_actions[run_id]  # shape (12, 2)

    print(f'\n[{run_id+1}/{n_trajectories}] Running trajectory {run_id}...')
    t0 = time.time()

    transitions, summary = run_single_trajectory(
        mygs, action_row, run_id, cwd, eqdsk_list, eqtimes,
        coil_bounds, x_points, diverted_isoflux_pts,
        Ip_targets, ne_init, Te_init, psi_sample,
    )

    elapsed = time.time() - t0

    if transitions is not None:
        path = save_trajectory(transitions, summary, action_row, run_id, output_dir)
        success_count += 1
        print(f'  Saved to {path} ({elapsed:.1f}s)')
    else:
        fail_count += 1
        fail_log = os.path.join(output_dir, 'failed_runs.txt')
        with open(fail_log, 'a') as f:
            f.write(f'{run_id}\n')
        print(f'  Failed run logged ({elapsed:.1f}s)')

    # Progress estimate
    elapsed_total = time.time() - t_start_total
    runs_done     = run_id - start_idx + 1
    avg_per_run   = elapsed_total / runs_done
    remaining     = (n_trajectories - run_id - 1) * avg_per_run
    print(f'  Progress: {success_count} ok, {fail_count} failed | '
          f'ETA: {remaining/60:.1f} min')

print(f'\nDone. {success_count}/{n_trajectories} successful trajectories saved to {output_dir}')

Sampling 600 trajectories with LHS (seed=42)
Action matrix saved: shape (600, 12, 2)
Setting up TokaMaker...
#----------------------------------------------
Open FUSION Toolkit Initialized
Development branch:    main
Revision id:           7bd3818
Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    2
Integer Precisions    =    4   8
Float Precisions      =    4   8  16
Complex Precisions    =    4   8
LA backend            = native
#----------------------------------------------


**** Loading OFT surface mesh

**** Generating surface grid level  1
  Generating boundary domain linkage
  Mesh statistics:
    Area         =  2.859E+02
    # of points  =    4757
    # of edges   =   14156
    # of cells   =    9400
    # of boundary points =     112
    # of boundary edges  =     112
    # of boundary cells  =     112
  Resolution statistics:
    hmin =  9.924E-03
    hrms =  2.826E-01
    hmax =  8.466E-01
  Surface grounded at vertex     870


**** Creating Lagrange

Simulating (t=5.00000): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:15<00:00,  6.38it/s]



  Loop 1
  TORAX: running simulation...


Simulating (t=600.00000): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:47<00:00,  2.12it/s]


  TORAX: done (cflux=114.1340 Wb)
  TokaMaker: solving 10 equilibria...


...

  TM loop 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:32<00:00,  3.23s/solve], t=600.00s OK(L0)


  TokaMaker: 10/10 solved (cflux=143.0458 Wb)
	TM summary: 10/10 solved. Levels: raw tx profs: 10.
  Loop 1 result: conv_err=0.000% | TX-TM diff=0.0004% | cflux_TX=143.0464 Wb | cflux_TM=143.0458 Wb

  Loop 2
  TORAX: Running relax (5 s) simulation...


Simulating (t=5.00000): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 28.56it/s]


  TORAX: running simulation...


Simulating (t=588.93034):  98%|████████████████████████████████████████████████████████████████████████████████████████████████  | 98/100 [00:05<00:00, 17.84it/s]

In [3]:
# Load and inspect the saved trajectory
import json

with open('./rl_dataset_test/trajectory_0000.json') as f:
    traj = json.load(f)

print("Keys:", traj.keys())
print("Summary:", traj['summary'])
print(f"\n{len(traj['transitions'])} transitions")
print(f"\nFirst transition state keys: {list(traj['transitions'][0]['s'].keys())}")

# Print the reward table
print(f"\n{'t':>6} {'t_next':>6} {'ecrh':>6} {'nbi':>6} {'reward':>8} {'Q_fusion':>9}")
for tr in traj['transitions']:
    print(f"{tr['t']:>6}  {tr['t_next']:>6}  {tr['a'][0]:>6.1f}  {tr['a'][1]:>6.1f}  "
          f"{tr['r']:>8.3f}  {tr['s']['tx_Q_fusion']:>9.3f}")

Keys: dict_keys(['run_id', 'timestamp', 'actions_raw', 'transitions', 'summary'])
Summary: {'Q_max': 3.755328268874835, 'Q_max_time': 380.8779949519889, 'Q_flattop_avg': 2.6928794639764955, 'E_fusion_total_MJ': 40748.91191640589, 'beta_N_max': 2.678675449759353, 'H98_max': 3.781003147321524, 'H98_flattop_avg': 0.6405578982286756, 'T_e_core_max_keV': 89.43045388170825, 'T_i_core_max_keV': 9.634845024088524, 'n_e_line_avg_max': 1.0084484175453422e+20, 'f_GW_max': 0.8422611865193793, 'q95_min': 2.7746145850112742, 'q0_min': 0.7493997538401009, 'flux_consumed_Wb': 107.73524630536632, 'P_fusion_max_MW': 129.9161299467818, 'P_ohmic_max_MW': 5.471633406667047, 'l_i_flattop_avg': 0.7542863911187893, 'vloop_tx_flattop_avg_V': 0.20469967702519087}

12 transitions

First transition state keys: ['tx_H98', 'tx_tau_E', 'tx_W_thermal_total', 'tx_P_SOL_total', 'tx_P_radiation_e', 'tx_P_aux_total', 'tx_f_non_inductive', 'tx_n_e_line_avg', 'tx_fgw_n_e_line_avg', 'tx_T_e_volume_avg', 'tx_T_i_volume_avg',

In [4]:
print(f"q95_min:    {traj['summary']['q95_min']:.3f}  (limit: {Q95_MIN})")
print(f"beta_N_max: {traj['summary']['beta_N_max']:.3f}  (limit: {BETA_N_MAX})")
print(f"f_GW_max:   {traj['summary']['f_GW_max']:.3f}  (limit: {FGW_MAX})")

print(f"\n{'t':>6} {'q95':>8} {'fgw':>8} {'beta_N':>8}")
print("-" * 35)
for tr in traj['transitions']:
    s = tr['s']
    print(f"{tr['t']:>6}  {s['tx_q95']:>8.3f}  {s['tx_fgw_n_e_line_avg']:>8.3f}  {s['tx_beta_N']:>8.3f}")

q95_min:    2.775  (limit: 3.0)
beta_N_max: 2.679  (limit: 2.8)
f_GW_max:   0.842  (limit: 0.85)

     t      q95      fgw   beta_N
-----------------------------------
    50     4.333     0.511     0.342
    70     4.207     0.520     0.217
    80     3.788     0.565     0.383
   130     3.214     0.789     0.704
   180     3.193     0.840     0.932
   230     3.179     0.828     0.914
   280     3.125     0.835     0.930
   330     2.968     0.827     0.913
   380     3.092     0.804     0.881
   450     3.100     0.852     0.990
   520     2.528     0.380     0.324
   580     2.569     0.214     0.765


In [5]:
import json

with open('./rl_dataset_test/trajectory_0000.json') as f:
    traj = json.load(f)

# Top level structure
print("Top-level keys:", list(traj.keys()))
print(f"Run ID: {traj['run_id']}")
print(f"Timestamp: {traj['timestamp']}")
print(f"Number of transitions: {len(traj['transitions'])}")
print(f"\nActions raw shape: {len(traj['actions_raw'])} x {len(traj['actions_raw'][0])}")

# Summary
print("\n--- Summary ---")
for k, v in traj['summary'].items():
    print(f"  {k:30s}: {v}")

# One transition in full detail
print("\n--- Transition 0 (t=50→70) ---")
tr = traj['transitions'][0]
print(f"  t={tr['t']} → t_next={tr['t_next']}")
print(f"  action: ecrh={tr['a'][0]:.2f} MW, nbi={tr['a'][1]:.2f} MW")
print(f"  reward: {tr['r']:.4f}")
print(f"  done: {tr['done']}")
print(f"\n  State variables:")
for k, v in tr['s'].items():
    print(f"    {k:35s}: {v:.4g}")

Top-level keys: ['run_id', 'timestamp', 'actions_raw', 'transitions', 'summary']
Run ID: 0
Timestamp: 2026-05-17T01:16:09.138357
Number of transitions: 12

Actions raw shape: 12 x 2

--- Summary ---
  Q_max                         : 3.755328268874835
  Q_max_time                    : 380.8779949519889
  Q_flattop_avg                 : 2.6928794639764955
  E_fusion_total_MJ             : 40748.91191640589
  beta_N_max                    : 2.678675449759353
  H98_max                       : 3.781003147321524
  H98_flattop_avg               : 0.6405578982286756
  T_e_core_max_keV              : 89.43045388170825
  T_i_core_max_keV              : 9.634845024088524
  n_e_line_avg_max              : 1.0084484175453422e+20
  f_GW_max                      : 0.8422611865193793
  q95_min                       : 2.7746145850112742
  q0_min                        : 0.7493997538401009
  flux_consumed_Wb              : 107.73524630536632
  P_fusion_max_MW               : 129.9161299467818
  P_ohmic_